# 05 · Secuencias y libro de órdenes — representación sí, política no

**Qué pregunta responde.** ¿Las variantes profundas del proyecto (GRU tabular, Conv1D book-only, fusión book+tabular, TCN/régimen) convierten su representación en una política económica estable bajo coste y validación temporal — o solo mejoran métricas aisladas?

**Qué entradas usa.** Solo CSV públicos del repositorio: `results/key_results.csv` (filas `sequence`, `orderbook`, `oof`), `final_candidate_summary.csv` y el ledger anonimizado. Los entrenamientos se hicieron en el workspace privado; aquí quedan scripts y resultados clave, no tensores ni modelos intermedios (dependen del corpus completo). Sin aleatoriedad: es un cuaderno de lectura re-ejecutable.

**Qué produce.** El resumen de la escalera profunda (arquitecturas evaluadas con su estado GO/NO_GO), las filas publicadas del registro y la tabla de afirmaciones-con-evidencia que justifica por qué el trabajo no termina en una red profunda.

**Conclusión (2 frases).** Las redes aportan representación pero ninguna produce una política estable con el soporte disponible: la GRU es inestable por días, el libro solo no generaliza (−1,083 ticks proxy @0,5) y la fusión no genera configuraciones robustas. Esto justifica volver a un especialista tabular más controlable para la decisión final (cuaderno 07).

**Filas de `results/key_results.csv`.** `sequence,tabular_gru`, `orderbook,tensor_audit`, `orderbook,book_only_conv1d` y `oof,clean_fusion_stage2` — las carga y muestra directamente la sección «Resultados publicados del bloque».

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

key = pd.read_csv(ROOT / 'results' / 'key_results.csv')
summary = pd.read_csv(ROOT / 'results' / 'final_candidate_summary.csv')
ledger = pd.read_csv(ROOT / 'results' / 'final_candidate_actions_anonymized.csv')

## Arquitecturas evaluadas

La escalera probó primero secuencias tabulares, después libro visible y finalmente fusión. La complejidad solo se acepta si mejora la política, no si mejora una métrica aislada.

In [2]:
pd.DataFrame([
    {"bloque": "GRU tabular", "idea": "Secuencia de features causales", "lectura": "Señal proxy positiva, pero con inestabilidad diaria", "estado": "NO_GO"},
    {"bloque": "Book-only Conv1D", "idea": "Tensor del libro visible sin features tabulares", "lectura": "El libro solo no generaliza en test", "estado": "NO_GO"},
    {"bloque": "Fusión book + tabular", "idea": "Combinar embedding del libro con secuencia tabular", "lectura": "Diagnóstico prometedor, no selector robusto", "estado": "NO_GO / diagnóstico"},
    {"bloque": "TCN / régimen", "idea": "Representar estados temporales y régimen", "lectura": "útil como pista de régimen, no como política final", "estado": "diagnóstico"},
])

,bloque,idea,lectura,estado
0,GRU tabular,Secuencia de features causales,"Señal proxy positiva, pero con inestabilidad d...",NO_GO
1,Book-only Conv1D,Tensor del libro visible sin features tabulares,El libro solo no generaliza en test,NO_GO
2,Fusión book + tabular,Combinar embedding del libro con secuencia tab...,"Diagnóstico prometedor, no selector robusto",NO_GO / diagnóstico
3,TCN / régimen,Representar estados temporales y régimen,"útil como pista de régimen, no como política f...",diagnóstico


## Resultados publicados del bloque

Estos son los resultados filtrados al arco principal que se conservan en `results/key_results.csv`. Sirven para justificar por qué el trabajo no termina en una red profunda.

In [3]:
deep_rows = key[key["phase"].isin(["sequence", "orderbook", "oof"])]
deep_rows[["phase", "experiment", "metric", "value", "unit", "status", "interpretation"]]

,phase,experiment,metric,value,unit,status,interpretation
5,sequence,tabular_gru,test_cost0p50_mean,0.4550,proxy_ticks,NO_GO,Signal remains but one bad test day
6,orderbook,tensor_audit,top10_complete,100,percent,GO,Orderbook tensor is model-ready
7,orderbook,book_only_conv1d,best_test_cost0p50_mean,-1.0830,proxy_ticks,NO_GO,Book alone is insufficient
9,oof,clean_h60_upstream,test_auc,0.5656,auc,PARTIAL_SIGNAL,Modest causal ranking signal
10,oof,clean_h60_upstream,test_top35_cost0p50_mean,0.1778,proxy_ticks,PARTIAL_SIGNAL,Positive under primary proxy cost
11,oof,clean_h60_upstream,test_top35_cost1p00_mean,-0.1899,proxy_ticks,NO_GO,Does not survive conservative cost
12,oof,clean_fusion_stage2,robust_configs,0,count,NO_GO,Second stage does not create stable policy


## Lectura

La conclusión es deliberadamente conservadora: las redes aportan representación, pero no una política estable con el soporte disponible. Esto justifica volver a un especialista tabular más controlable para la decisión final.

In [4]:
pd.DataFrame([
    {"afirmación": "El deep learning fue evaluado", "evidencia": "GRU, Conv1D, fusión y régimen aparecen en scripts/docs/resultados"},
    {"afirmación": "No se promociona por estética", "evidencia": "book_only_conv1d queda en -1.083 ticks proxy @0.5 y la fusión no produce configs robustas"},
    {"afirmación": "La señal útil se mantiene como hipótesis", "evidencia": "H60 conserva señal parcial, pero no coste conservador ni estabilidad suficiente"},
])

,afirmación,evidencia
0,El deep learning fue evaluado,"GRU, Conv1D, fusión y régimen aparecen en scri..."
1,No se promociona por estética,book_only_conv1d queda en -1.083 ticks proxy @...
2,La señal útil se mantiene como hipótesis,"H60 conserva señal parcial, pero no coste cons..."
